# Lecture 12 Activity – Who wrote more of the joint essays?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

Federalist 18, 19, and 20 compare the proposed Constitution with earlier
confederacies: the Greek leagues, the German empire, and the Dutch
provinces. Hamilton and Madison wrote the three together, from notes both
had collected. The lecture settled the 12 disputed essays with two routes:
similarity to each author's essays by subject words, and the rates of style
words such as `upon` and `whilst`.

> **In each of the three joint essays, whose hand shows more, and how far
> can word counts settle that?**

### How to work

Work in pairs for 30 minutes. Submit the nearest essays of each joint essay,
their similarity to each author, their style rates, their topic, and a short
claim for each essay. If you run out of time, finish Task 5 after class.

- The row unit is **one essay**. The setup reads the 85 files and labels the
  authors as the lecture did.
- Use the lecture's settings: `stop_words='english'`, `min_df=2`, four
  topics with `random_state=0`.
- In the starter cells, replace each `...` with code.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import (
    CountVectorizer, TfidfVectorizer)
from sklearn.metrics.pairwise import cosine_similarity

RAW = Path('../data/raw')

rows = []
for n in range(1, 86):
    path = RAW / 'federalist' / f'fp{n:02d}.txt'
    rows.append({'essay': n,
                 'text': path.read_text(encoding='utf-8')})
essays = pd.DataFrame(rows)

authors = {
    'Hamilton': ([1, 6, 7, 8, 9, 11, 12, 13, 15, 16, 17]
                 + list(range(21, 37)) + [59, 60, 61]
                 + list(range(65, 86))),
    'Madison': [10, 14] + list(range(37, 49)),
    'Jay': [2, 3, 4, 5, 64],
    'Joint': [18, 19, 20],
    'Disputed': list(range(49, 59)) + [62, 63],
}
label = {n: author for author, ns in authors.items() for n in ns}
essays['author'] = essays['essay'].map(label)
counts = CountVectorizer().fit_transform(essays['text'])
essays['n_words'] = np.asarray(counts.sum(axis=1)).ravel()

# Check: 85 essays, three of them joint.
essays['author'].value_counts()

author
Hamilton    51
Madison     14
Disputed    12
Jay          5
Joint        3
Name: count, dtype: int64

## Task 1 · The nearest essays of each joint essay

Build the tf-idf matrix with the lecture's settings and the 85 by 85 cosine
similarity table, with essay numbers as row and column labels. For each of
essays 18, 19, and 20, list the five nearest essays (not itself) with their
similarity and author.

Do the neighbors come from one author?

In [2]:
tfidf_vec = TfidfVectorizer(stop_words='english', min_df=2)
tfidf = tfidf_vec.fit_transform(essays['text'])
terms = tfidf_vec.get_feature_names_out()
sim = pd.DataFrame(cosine_similarity(tfidf),
                   index=essays['essay'], columns=essays['essay'])
sim.shape

(85, 85)

In [3]:
for n in [18, 19, 20]:
    nearest = sim[n].drop(n).sort_values(ascending=False).head(5)
    print(pd.DataFrame({'similarity': nearest.round(3),
                        'author': nearest.index.map(label)}), '\n')

       similarity    author
essay                      
9           0.197  Hamilton
20          0.194     Joint
43          0.172   Madison
38          0.166   Madison
6           0.160  Hamilton 

       similarity    author
essay                      
22          0.191  Hamilton
42          0.183   Madison
6           0.180  Hamilton
15          0.179  Hamilton
45          0.176   Madison 

       similarity    author
essay                      
15          0.232  Hamilton
22          0.216  Hamilton
45          0.205   Madison
43          0.205   Madison
18          0.194     Joint 



The neighbors are mixed. Essay 18's nearest are 9 (Hamilton), 20 (joint),
43 and 38 (Madison), and 6 (Hamilton). Essay 19's are 22, 6, and 15
(Hamilton) with 42 and 45 (Madison). Essay 20's are 15 and 22 (Hamilton),
45 and 43 (Madison), and 18 (joint). The three essays sit among the essays
on the union and the defects of confederation, which both men wrote.

## Task 2 · Similarity to each author's essays

For every essay, compute the mean similarity to Hamilton's 51 essays and to
Madison's 14, leaving each essay's similarity to itself out. Show the two
numbers for essays 18, 19, and 20, and the difference between them.

Which author is each joint essay closer to, and is the margin large next to
the margins of the 12 disputed essays in the lecture (0.03 to 0.07)?

In [4]:
known = sim.copy()
np.fill_diagonal(known.values, np.nan)
essays['to_hamilton'] = known[authors['Hamilton']].mean(axis=1).values
essays['to_madison'] = known[authors['Madison']].mean(axis=1).values
essays['margin'] = essays['to_madison'] - essays['to_hamilton']
joint = essays[essays['author'] == 'Joint']
joint[['essay', 'to_hamilton', 'to_madison', 'margin']].round(3)

,essay,to_hamilton,to_madison,margin
17,18,0.092,0.117,0.025
18,19,0.112,0.132,0.020
19,20,0.132,0.155,0.022


All three are closer to Madison's essays, by about 0.02 in each case (18:
0.117 against 0.092; 19: 0.132 against 0.112; 20: 0.155 against 0.132).
The margins are smaller than those of most disputed essays, and the lecture
showed that this route misfiles 12 of Hamilton's own essays, so a margin of
0.02 is weak evidence.

## Task 3 · The style words

Count `upon`, `whilst`, `while`, and `on` in every essay with a fixed
vocabulary, and turn the counts into rates per 1,000 words. Show the raw
counts and the rates for essays 18, 19, and 20, then the mean rates by
author for comparison.

Which way does each essay lean, and how many occurrences is that lean built
on?

In [5]:
style_vec = CountVectorizer(
    vocabulary=['upon', 'whilst', 'while', 'on'])
style_matrix = style_vec.fit_transform(essays['text']).toarray()
style_counts = pd.DataFrame(style_matrix,
                            columns=style_vec.get_feature_names_out())
style = style_counts.div(essays['n_words'], axis=0) * 1000
style['author'] = essays['author']
is_joint = essays['author'] == 'Joint'
joint_numbers = essays.loc[is_joint, 'essay']
print(style_counts[is_joint].set_index(joint_numbers))
style[is_joint].set_index(joint_numbers).round(2)

       upon  whilst  while  on
essay                         
18        1       1      0  15
19        0       1      0  17
20        1       0      0   7


,upon,whilst,while,on,author
essay,,,,,
18,0.49,0.49,0.0,7.32,Joint
19,0.00,0.50,0.0,8.52,Joint
20,0.67,0.00,0.0,4.69,Joint


In [6]:
style.groupby('author').mean().round(2)

,upon,whilst,while,on
author,,,,
Disputed,0.16,0.38,0.00,8.32
Hamilton,3.38,0.01,0.28,3.42
Jay,0.12,0.00,0.21,5.17
Joint,0.39,0.33,0.00,6.84
Madison,0.17,0.32,0.00,7.81


Essay 18 has one `upon` and one `whilst`: one Hamilton marker and one
Madison marker. Essay 19 has one `whilst` and no `upon`, which leans
Madison. Essay 20 has one `upon` and no `whilst`, which leans Hamilton. All
three use `on` at 5 to 9 per 1,000 words, Madison's rate (7.9) rather than
Hamilton's (3.4). Each lean rests on a single occurrence in about 2,000
words; a Hamilton essay has `upon` 3.4 times per 1,000, so even essay 20's
one `upon` is far below his rate.

## Task 4 · The topic of the three essays

Fit `NMF` with four topics on the tf-idf matrix, as in the lecture. Give
each essay its largest topic, print the ten strongest words of the topic
that holds essays 18 to 20, and print the eight strongest tf-idf words of
each of the three essays.

Why can the topic not decide who wrote more of an essay?

In [7]:
nmf = NMF(n_components=4, random_state=0, max_iter=1000)
weights = nmf.fit_transform(tfidf)
essays['topic'] = weights.argmax(axis=1)
essays.loc[is_joint, ['essay', 'topic']]

,essay,topic
17,18,0
18,19,0
19,20,0


In [8]:
topic = essays.loc[essays['essay'] == 18, 'topic'].iloc[0]
top = nmf.components_[topic].argsort()[::-1][:10]
print(topic, ', '.join(terms[top]))

0 government, states, union, national, nations, people, war, state, power, peace


In [9]:
for n in [18, 19, 20]:
    row = tfidf[n - 1].toarray().ravel()
    print(n, ', '.join(terms[row.argsort()[::-1][:8]]))

18 cities, macedon, achaeans, greece, league, members, sparta, athens
19 emperor, empire, diet, imperial, cantons, princes, city, members
20 provinces, says, province, union, foreign, treaty, vices, cities


All three essays fall in topic 0, the union and foreign nations
(`government`, `states`, `union`, `national`, `nations`, `war`). Their own
strongest words are the confederacies they describe: `macedon`, `achaeans`,
`greece` (18); `emperor`, `empire`, `diet`, `cantons` (19); `provinces`,
`union`, `treaty` (20). A topic names the subject of an essay. Hamilton has
25 essays in this topic and Madison 3, which says who wrote more often on
the union, not who wrote these three.

## Task 5 · Make a claim for each essay

Write three or four sentences. For each of essays 18, 19, and 20, say whose
hand the counts favor and which piece of evidence you trust most. Then say
why an answer that names a single author is wrong by construction for these
three essays, and what the counts can and cannot show about a text two
people wrote.

The subject route favors Madison for all three essays by a margin of 0.02,
too small to trust after the lecture showed it misfiles nearly a quarter
of Hamilton's essays. The style route rests on one occurrence per essay: 19
leans Madison (`whilst`), 20 leans Hamilton (`upon`), and 18 splits. The `on`
rate, 5 to 9 per 1,000 words, is the most Madison-like signal in all three.
A single-author answer is wrong by construction because both men wrote the
essays: the historical record says Madison drafted them from notes both had
collected, so the counts describe a blend. Word counts can say which
author's habits a text resembles more; they cannot divide one essay into
the sentences each man wrote.

## Checks for a claim built on counts

- Report the count behind every rate. A rate of 0.5 per 1,000 words built
  on one occurrence is one data point.
- Test a rule on the cases with known answers before using it on the open
  ones. The subject route misfiled 12 of 51 Hamilton essays.
- Say what the representation measures: tf-idf and topics measure subject,
  the style words measure habit. Match the measure to the question.